In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"
for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0, REPO) if REPO not in sys.path else None
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Mounted at /content/drive
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy pyarrow requests --quiet

In [3]:
# Add fam_train to src/data.py and write the new src/tsfs.py module
import base64, os
os.makedirs("src", exist_ok=True)
open("src/data.py", "w").write(base64.b64decode("IiIiRGF0YSBsb2FkaW5nLCBzY2hlbWEgZGV0ZWN0aW9uLCBhbmQgYXR0YWNrLWZhbWlseS1oZWxkLW91dCBzcGxpdHRpbmcgZm9yClVBViBuZXR3b3JrLUlEUyBkYXRhc2V0cy4gRGF0YXNldC1hZ25vc3RpYyBzbyBvbmUgcGlwZWxpbmUgc2VydmVzIFVBVklEUy0yMDI1CmFuZCB0aGUgY29tcGFuaW9uIGRhdGFzZXRzLgoiIiIKaW1wb3J0IGdsb2IKaW1wb3J0IG9zCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlcgoKS05PV05fQ0xBU1NFUyA9IHsibm9ybWFsIiwgImJsYWNraG9sZSIsICJmbG9vZGluZyIsICJzeWJpbCIsICJ3b3JtaG9sZSIsCiAgICAgICAgICAgICAgICAgImJlbmlnbiIsICJhdHRhY2siLCAiZG9zIiwgImRkb3MiLCAicmVwbGF5IiwgImZ1enp5In0KTEFCRUxfTkFNRVMgPSB7ImxhYmVsIiwgImNsYXNzIiwgImF0dGFjayIsICJjYXRlZ29yeSIsICJ0eXBlIiwgInRhcmdldCIsCiAgICAgICAgICAgICAgICJhdHRhY2tfdHlwZSIsICJ0cmFmZmljX3R5cGUifQpOT1JNQUxfTkFNRVMgPSB7Im5vcm1hbCIsICJiZW5pZ24iLCAiMCIsICJub25lIiwgImNsZWFuIn0KCgpkZWYgbG9hZF9jc3ZzKGRhdGFfZGlyKToKICAgICIiIkxvYWQgYW5kIGNvbmNhdGVuYXRlIENTVnMgdW5kZXIgZGF0YV9kaXIgdGhhdCBzaGFyZSB0aGUgd2lkZXN0IHNjaGVtYS4iIiIKICAgIGNzdnMgPSBnbG9iLmdsb2Iob3MucGF0aC5qb2luKGRhdGFfZGlyLCAiKiovKi5jc3YiKSwgcmVjdXJzaXZlPVRydWUpCiAgICBpZiBub3QgY3N2czoKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigiTm8gQ1NWIGZvdW5kIHVuZGVyICIgKyBkYXRhX2RpcikKICAgIGZyYW1lcyA9IFtwZC5yZWFkX2NzdihjLCBsb3dfbWVtb3J5PUZhbHNlKSBmb3IgYyBpbiBjc3ZzXQogICAgd2lkZXN0ID0gbWF4KGZyYW1lcywga2V5PWxhbWJkYSBkOiBkLnNoYXBlWzFdKS5jb2x1bW5zCiAgICBrZWVwID0gW2YgZm9yIGYgaW4gZnJhbWVzIGlmIHNldCh3aWRlc3QpLmlzc3Vic2V0KGYuY29sdW1ucyldCiAgICBkZiA9IHBkLmNvbmNhdChrZWVwLCBpZ25vcmVfaW5kZXg9VHJ1ZSkgaWYgbGVuKGtlZXApID4gMSBlbHNlIGZyYW1lc1swXQogICAgZGYuY29sdW1ucyA9IFtzdHIoYykuc3RyaXAoKSBmb3IgYyBpbiBkZi5jb2x1bW5zXQogICAgcmV0dXJuIGRmCgoKZGVmIGRldGVjdF9zY2hlbWEoZGYsIGxhYmVsX2NvbD1Ob25lLCBub3JtYWxfdmFsdWU9Tm9uZSk6CiAgICAiIiJSZXR1cm4gKGxhYmVsX2NvbCwgbm9ybWFsX3ZhbHVlLCBmYW1pbGllcyksIGF1dG8tZGV0ZWN0aW5nIHdoZXJlIG5vdCBnaXZlbi4iIiIKICAgIGlmIGxhYmVsX2NvbCBpcyBOb25lOgogICAgICAgIGNhbmRzID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjLmxvd2VyKCkgaW4gTEFCRUxfTkFNRVNdCiAgICAgICAgaWYgbm90IGNhbmRzOgogICAgICAgICAgICBmb3IgYyBpbiBkZi5jb2x1bW5zOgogICAgICAgICAgICAgICAgdmFscyA9IHNldChzdHIodikuc3RyaXAoKS5sb3dlcigpIGZvciB2IGluIGRmW2NdLmRyb3BuYSgpLnVuaXF1ZSgpWzo1MF0pCiAgICAgICAgICAgICAgICBpZiBsZW4odmFscyAmIEtOT1dOX0NMQVNTRVMpID49IDI6CiAgICAgICAgICAgICAgICAgICAgY2FuZHMuYXBwZW5kKGMpCiAgICAgICAgaWYgbm90IGNhbmRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDb3VsZCBub3QgZGV0ZWN0IGxhYmVsIGNvbHVtbjsgcGFzcyBsYWJlbF9jb2wgZXhwbGljaXRseS4iKQogICAgICAgIGxhYmVsX2NvbCA9IGNhbmRzWzBdCiAgICBpZiBub3JtYWxfdmFsdWUgaXMgTm9uZToKICAgICAgICBmb3IgdiBpbiBkZltsYWJlbF9jb2xdLnVuaXF1ZSgpOgogICAgICAgICAgICBpZiBzdHIodikuc3RyaXAoKS5sb3dlcigpIGluIE5PUk1BTF9OQU1FUzoKICAgICAgICAgICAgICAgIG5vcm1hbF92YWx1ZSA9IHYKICAgICAgICBpZiBub3JtYWxfdmFsdWUgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ291bGQgbm90IGRldGVjdCBOb3JtYWwgdmFsdWU7IHBhc3Mgbm9ybWFsX3ZhbHVlIGV4cGxpY2l0bHkuIikKICAgIGZhbWlsaWVzID0gW3YgZm9yIHYgaW4gZGZbbGFiZWxfY29sXS51bmlxdWUoKSBpZiB2ICE9IG5vcm1hbF92YWx1ZV0KICAgIHJldHVybiBsYWJlbF9jb2wsIG5vcm1hbF92YWx1ZSwgZmFtaWxpZXMKCgpkZWYgX3NwbGl0X2lkeChpZHgsIGZyYWNzLCBzZWVkKToKICAgIGlkeCA9IG5wLmFycmF5KGlkeCk7IHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKTsgcm5nLnNodWZmbGUoaWR4KQogICAgb3V0LCBzdGFydCwga2V5cyA9IHt9LCAwLCBsaXN0KGZyYWNzKQogICAgZm9yIGksIGsgaW4gZW51bWVyYXRlKGtleXMpOgogICAgICAgIHN0b3AgPSBsZW4oaWR4KSBpZiBpID09IGxlbihrZXlzKSAtIDEgZWxzZSBzdGFydCArIGludChyb3VuZChmcmFjc1trXSAqIGxlbihpZHgpKSkKICAgICAgICBvdXRba10gPSBpZHhbc3RhcnQ6c3RvcF07IHN0YXJ0ID0gc3RvcAogICAgcmV0dXJuIG91dAoKCmRlZiBwcmVwYXJlX3NwbGl0cyhkZiwgbGFiZWxfY29sLCBub3JtYWxfdmFsdWUsIGhlbGRfb3V0X2ZhbWlseSwKICAgICAgICAgICAgICAgICAgIGRyb3BfcGF0dGVybnMsIG5vcm1hbF9mcmFjcywgZmFtaWx5X2ZyYWNzLCBzZWVkKToKICAgICIiIkJ1aWxkIHRoZSBiaW5hcnkgbGFiZWwgYW5kIGF0dGFjay1mYW1pbHktaGVsZC1vdXQgc3BsaXRzLgoKICAgIFJldHVybnMgYSBkaWN0IHdpdGggc2NhbGVkIGZlYXR1cmUgbWF0cmljZXMgYW5kIGxhYmVscyBmb3IgdHJhaW4sIGNhbCwKICAgIHNlZW4tdGVzdCBhbmQgc2hpZnQtdGVzdCwgcGVyLXNhbXBsZSBmYW1pbHkgbGFiZWxzIChpbmNsdWRpbmcgZmFtX3RyYWluKSwKICAgIGFuZCBtZXRhZGF0YSAocmVzb2x2ZWQgaGVsZC1vdXQgZmFtaWx5LCBzZWVuIGZhbWlsaWVzLCBkcm9wcGVkIGNvbHVtbnMpLgogICAgIiIiCiAgICBmYW1pbGllcyA9IFt2IGZvciB2IGluIGRmW2xhYmVsX2NvbF0udW5pcXVlKCkgaWYgdiAhPSBub3JtYWxfdmFsdWVdCiAgICBtYXRjaCA9IFtmIGZvciBmIGluIGZhbWlsaWVzCiAgICAgICAgICAgICBpZiBzdHIoZikuc3RyaXAoKS5sb3dlcigpID09IHN0cihoZWxkX291dF9mYW1pbHkpLnN0cmlwKCkubG93ZXIoKV0KICAgIGhlbGRfb3V0ID0gbWF0Y2hbMF0gaWYgbWF0Y2ggZWxzZSBzb3J0ZWQoZmFtaWxpZXMsIGtleT1sYW1iZGEgZjogKGRmW2xhYmVsX2NvbF0gPT0gZikuc3VtKCkpWzBdCiAgICBzZWVuX2ZhbWlsaWVzID0gW2YgZm9yIGYgaW4gZmFtaWxpZXMgaWYgZiAhPSBoZWxkX291dF0KCiAgICBkcm9wX2NvbHMgPSBbYyBmb3IgYyBpbiBkZi5jb2x1bW5zIGlmIGMgIT0gbGFiZWxfY29sCiAgICAgICAgICAgICAgICAgYW5kIGFueShwIGluIGMubG93ZXIoKSBmb3IgcCBpbiBkcm9wX3BhdHRlcm5zKV0KICAgIGZlYXQgPSBkZi5kcm9wKGNvbHVtbnM9ZHJvcF9jb2xzICsgW2xhYmVsX2NvbF0pCiAgICBjb25zdCA9IFtjIGZvciBjIGluIGZlYXQuY29sdW1ucyBpZiBmZWF0W2NdLm51bmlxdWUoZHJvcG5hPUZhbHNlKSA8PSAxXQogICAgZmVhdCA9IGZlYXQuZHJvcChjb2x1bW5zPWNvbnN0KQogICAgY2F0ID0gW2MgZm9yIGMgaW4gZmVhdC5jb2x1bW5zIGlmIG5vdCBwZC5hcGkudHlwZXMuaXNfbnVtZXJpY19kdHlwZShmZWF0W2NdKV0KICAgIGZlYXQgPSBwZC5nZXRfZHVtbWllcyhmZWF0LCBjb2x1bW5zPWNhdCwgZHVtbXlfbmE9RmFsc2UpCiAgICBmZWF0ID0gZmVhdC5yZXBsYWNlKFtucC5pbmYsIC1ucC5pbmZdLCBucC5uYW4pLmZpbGxuYSgwLjApCgogICAgeSA9IChkZltsYWJlbF9jb2xdLnZhbHVlcyAhPSBub3JtYWxfdmFsdWUpLmFzdHlwZShpbnQpCiAgICBYID0gZmVhdC52YWx1ZXMuYXN0eXBlKGZsb2F0KQogICAgZmFtID0gZGZbbGFiZWxfY29sXS52YWx1ZXMKCiAgICB0ciwgY2EsIHNlZW4sIHNoaWZ0ID0gW10sIFtdLCBbXSwgW10KICAgIG5zID0gX3NwbGl0X2lkeChucC53aGVyZShmYW0gPT0gbm9ybWFsX3ZhbHVlKVswXSwgbm9ybWFsX2ZyYWNzLCBzZWVkKQogICAgdHIgKz0gbGlzdChuc1sidHJhaW4iXSk7IGNhICs9IGxpc3QobnNbImNhbCJdKQogICAgc2VlbiArPSBsaXN0KG5zWyJ0ZXN0X3NlZW4iXSk7IHNoaWZ0ICs9IGxpc3QobnNbInRlc3Rfc2hpZnQiXSkKICAgIGZvciBqLCBmIGluIGVudW1lcmF0ZShzZWVuX2ZhbWlsaWVzKToKICAgICAgICBmcyA9IF9zcGxpdF9pZHgobnAud2hlcmUoZmFtID09IGYpWzBdLCBmYW1pbHlfZnJhY3MsIHNlZWQgKyBqICsgMSkKICAgICAgICB0ciArPSBsaXN0KGZzWyJ0cmFpbiJdKTsgY2EgKz0gbGlzdChmc1siY2FsIl0pOyBzZWVuICs9IGxpc3QoZnNbInRlc3Rfc2VlbiJdKQogICAgc2hpZnQgKz0gbGlzdChucC53aGVyZShmYW0gPT0gaGVsZF9vdXQpWzBdKQogICAgdHIsIGNhLCBzZWVuLCBzaGlmdCA9IChucC5hcnJheShzb3J0ZWQoYSkpIGZvciBhIGluICh0ciwgY2EsIHNlZW4sIHNoaWZ0KSkKCiAgICBzY2FsZXIgPSBTdGFuZGFyZFNjYWxlcigpLmZpdChYW3RyXSkKICAgIHRmID0gbGFtYmRhIGl4OiBzY2FsZXIudHJhbnNmb3JtKFhbaXhdKQogICAgcmV0dXJuIHsKICAgICAgICAiaGVsZF9vdXQiOiBoZWxkX291dCwgInNlZW5fZmFtaWxpZXMiOiBzZWVuX2ZhbWlsaWVzLAogICAgICAgICJkcm9wcGVkIjogeyJpZF9sZWFrYWdlIjogZHJvcF9jb2xzLCAiY29uc3RhbnQiOiBjb25zdCwgImVuY29kZWQiOiBjYXR9LAogICAgICAgICJmZWF0dXJlX25hbWVzIjogbGlzdChmZWF0LmNvbHVtbnMpLAogICAgICAgICJYX3RyYWluIjogdGYodHIpLCAieV90cmFpbiI6IHlbdHJdLCAiZmFtX3RyYWluIjogZmFtW3RyXSwKICAgICAgICAiWF9jYWwiOiB0ZihjYSksICJ5X2NhbCI6IHlbY2FdLAogICAgICAgICJYX3NlZW4iOiB0ZihzZWVuKSwgInlfc2VlbiI6IHlbc2Vlbl0sICJmYW1fc2VlbiI6IGZhbVtzZWVuXSwKICAgICAgICAiWF9zaGlmdCI6IHRmKHNoaWZ0KSwgInlfc2hpZnQiOiB5W3NoaWZ0XSwgImZhbV9zaGlmdCI6IGZhbVtzaGlmdF0sCiAgICAgICAgIm4iOiB7InRyYWluIjogbGVuKHRyKSwgImNhbCI6IGxlbihjYSksICJzZWVuIjogbGVuKHNlZW4pLCAic2hpZnQiOiBsZW4oc2hpZnQpfSwKICAgIH0K").decode())
open("src/tsfs.py", "w").write(base64.b64decode("IiIiVHJhbnNmZXItU3RhYmxlIEZlYXR1cmUgU2VsZWN0aW9uIChUU0ZTKS4KClNlbGVjdHMgZmVhdHVyZXMgdGhhdCBnZW5lcmFsaXplIGFjcm9zcyBhdHRhY2sgZmFtaWxpZXMgdXNpbmcgT05MWSB0aGUgc2VlbgpmYW1pbGllcy4gQSBmZWF0dXJlIGlzIHRyYW5zZmVyLWZyYWdpbGUgaWYgaXRzIFNIQVAgYXR0cmlidXRpb24gcmV2ZXJzZXMgZnJvbQpwcm8tYXR0YWNrIHRvIHByby1ub3JtYWwgb24gYW4gdW5zZWVuIGZhbWlseTsgVFNGUyBlc3RpbWF0ZXMgdGhpcyB3aXRob3V0IHRoZQp1bnNlZW4gZmFtaWx5LCB2aWEgYW4gaW50ZXJuYWwgbGVhdmUtb25lLXNlZW4tZmFtaWx5LW91dCBsb29wLCBhbmQgZHJvcHMgdGhlCm1vc3QgZnJhZ2lsZSBmZWF0dXJlcyBiZWZvcmUgdHJhaW5pbmcgdGhlIGZpbmFsIGRldGVjdG9yLiBUaGlzIHR1cm5zIHRoZQpmcmFnaWxpdHkgZGlhZ25vc2lzIGludG8gYSBwcmFjdGljYWwsIGxlYWthZ2UtZnJlZSBkZWZlbmNlIGFnYWluc3QKYXR0YWNrLWZhbWlseSBkaXN0cmlidXRpb24gc2hpZnQuCiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHNoYXAKCgpkZWYgbWVhbl9zaGFwKGV4cGxhaW5lciwgWCk6CiAgICAiIiJNZWFuIHBlci1mZWF0dXJlIFNIQVAgdmFsdWUgb24gYSBzYW1wbGUgKGJpbmFyeTsgcG9zaXRpdmUgLT4gYXR0YWNrIGNsYXNzKS4iIiIKICAgIHMgPSBucC5hc2FycmF5KGV4cGxhaW5lci5zaGFwX3ZhbHVlcyhYKSkKICAgIGlmIHMubmRpbSA9PSAzOgogICAgICAgIHMgPSBzWy4uLiwgMV0KICAgIHJldHVybiBzLm1lYW4oMCkKCgpkZWYgcmV2ZXJzYWxfdmVjdG9yKG1fcmVmZXJlbmNlLCBtX3RhcmdldCk6CiAgICAiIiJQZXItZmVhdHVyZSBhdHRyaWJ1dGlvbiByZXZlcnNhbDogcHJvLWF0dGFjayBvbiByZWZlcmVuY2UsIHByby1ub3JtYWwgb24gdGFyZ2V0LiIiIgogICAgcmV0dXJuIG5wLm1heGltdW0obnAuYXNhcnJheShtX3JlZmVyZW5jZSksIDAuMCkgKiBucC5tYXhpbXVtKC1ucC5hc2FycmF5KG1fdGFyZ2V0KSwgMC4wKQoKCmRlZiBpbnRlcm5hbF9mcmFnaWxpdHkoZml0X21vZGVsLCBYLCB5LCBmYW0sIG5vcm1hbF92YWx1ZSwgc2Vlbl9mYW1pbGllcywKICAgICAgICAgICAgICAgICAgICAgICBuX2ZlYXR1cmVzLCBuX3NoYXA9MjAwMCwgc2VlZD0wKToKICAgICIiIkVzdGltYXRlIHBlci1mZWF0dXJlIHRyYW5zZmVyLWZyYWdpbGl0eSB1c2luZyBvbmx5IHRoZSBzZWVuIGZhbWlsaWVzLgoKICAgIGZpdF9tb2RlbChYX3RyLCB5X3RyLCBzZWVkKSBtdXN0IHJldHVybiBhIGZpdHRlZCB0cmVlIGNsYXNzaWZpZXIgd2l0aAogICAgcHJlZGljdF9wcm9iYS4gRm9yIGVhY2ggc2VlbiBmYW1pbHkgZiwgYSBtb2RlbCBpcyB0cmFpbmVkIG9uIG5vcm1hbCB0cmFmZmljCiAgICBwbHVzIHRoZSBPVEhFUiBzZWVuIGZhbWlsaWVzLCBhbmQgdGhlIHJldmVyc2FsIG9mIGVhY2ggZmVhdHVyZSdzIGF0dHJpYnV0aW9uCiAgICBvbiBmIChhIGhlbGQtaW4gcHJveHkgZm9yIGFuIHVuc2VlbiBmYW1pbHkpIGlzIGFjY3VtdWxhdGVkLiBUaGUgdW5zZWVuIHRlc3QKICAgIGZhbWlseSBuZXZlciBlbnRlcnMgdGhpcyBjb21wdXRhdGlvbi4KICAgICIiIgogICAgZmFtID0gbnAuYXNhcnJheShmYW0pCiAgICBhY2MgPSBucC56ZXJvcyhuX2ZlYXR1cmVzKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBmb3IgaSwgZiBpbiBlbnVtZXJhdGUoc2Vlbl9mYW1pbGllcyk6CiAgICAgICAgb3RoZXJzID0gW2cgZm9yIGcgaW4gc2Vlbl9mYW1pbGllcyBpZiBnICE9IGZdCiAgICAgICAgdHIgPSAoZmFtID09IG5vcm1hbF92YWx1ZSkgfCBucC5pc2luKGZhbSwgb3RoZXJzKQogICAgICAgIGNsZiA9IGZpdF9tb2RlbChYW3RyXSwgeVt0cl0sIHNlZWQgKyBpKQogICAgICAgIGV4cGwgPSBzaGFwLlRyZWVFeHBsYWluZXIoY2xmKQoKICAgICAgICBkZWYgc2FtcChtYXNrKToKICAgICAgICAgICAgaWR4ID0gbnAud2hlcmUobWFzaylbMF0KICAgICAgICAgICAgaWYgbGVuKGlkeCkgPiBuX3NoYXA6CiAgICAgICAgICAgICAgICBpZHggPSBybmcuY2hvaWNlKGlkeCwgbl9zaGFwLCByZXBsYWNlPUZhbHNlKQogICAgICAgICAgICByZXR1cm4gWFtpZHhdCgogICAgICAgIG1fcmVmID0gbWVhbl9zaGFwKGV4cGwsIHNhbXAobnAuaXNpbihmYW0sIG90aGVycykpKQogICAgICAgIG1fZiA9IG1lYW5fc2hhcChleHBsLCBzYW1wKGZhbSA9PSBmKSkKICAgICAgICBhY2MgKz0gcmV2ZXJzYWxfdmVjdG9yKG1fcmVmLCBtX2YpCiAgICByZXR1cm4gYWNjIC8gbWF4KGxlbihzZWVuX2ZhbWlsaWVzKSwgMSkKCgpkZWYgc2VsZWN0X3N0YWJsZV9mZWF0dXJlcyhmcmFnaWxpdHksIGRyb3BfZnJhY3Rpb249MC4yKToKICAgICIiIlJldHVybiBpbmRpY2VzIG9mIHRoZSB0cmFuc2Zlci1zdGFibGUgZmVhdHVyZXMgKGRyb3AgdGhlIG1vc3QtZnJhZ2lsZSBmcmFjdGlvbikuIiIiCiAgICBmcmFnaWxpdHkgPSBucC5hc2FycmF5KGZyYWdpbGl0eSkKICAgIHAgPSBsZW4oZnJhZ2lsaXR5KQogICAgbl9kcm9wID0gaW50KHJvdW5kKGRyb3BfZnJhY3Rpb24gKiBwKSkKICAgIGlmIG5fZHJvcCA8PSAwOgogICAgICAgIHJldHVybiBucC5hcmFuZ2UocCkKICAgIGRyb3AgPSBzZXQobnAuYXJnc29ydCgtZnJhZ2lsaXR5KVs6bl9kcm9wXS50b2xpc3QoKSkKICAgIHJldHVybiBucC5hcnJheShbaiBmb3IgaiBpbiByYW5nZShwKSBpZiBqIG5vdCBpbiBkcm9wXSkK").decode())
print("wrote src/data.py (updated) and src/tsfs.py")

wrote src/data.py (updated) and src/tsfs.py


In [4]:
# Imports (reload modules in case they were already imported this session)
import importlib, src.data, src.tsfs
importlib.reload(src.data); importlib.reload(src.tsfs)
import numpy as np, pandas as pd, requests, glob, zipfile
import matplotlib.pyplot as plt
import xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.data import load_csvs, detect_schema, prepare_splits
from src.tsfs import mean_shap, reversal_vector, internal_fragility, select_stable_features
from src.trust import top_label_ece, conformal_qhat
print("imports ready")

imports ready


In [5]:
# Config: both datasets, TSFS drop-fraction sweep
DATASETS = [
  {"name": "UAVIDS-2025", "kind": "zenodo", "record": "15336998",
   "data_dir": "data/uavids2025", "label_col": "label", "normal_value": "Normal Traffic",
   "include_families": None, "subsample_n": None,
   "drops": ["unnamed","flowid","srcaddr","dstaddr","srcport","dstport","index","timestamp"]},
  {"name": "UAV-NIDD", "kind": "file",
   "file": "data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv", "parquet": "data/uav_nidd/case1.parquet",
   "data_dir": "data/uav_nidd", "label_col": "Label", "normal_value": "Normal",
   "include_families": ["DDoS","UDP Flooding","MITM","Jamming","BruteForce","De-authentication"],
   "subsample_n": 200000,
   "drops": ["unnamed","index","ip.src","ip.dst","srcport","dstport","udp.srcport","udp.dstport",
             "frame.time","frame.number","time_epoch","time_relative","time_delta","bssid","mactime",
             "vendor_oui","wlan_radio.timestamp","wlan_radio.start_tsf","radiotap.timestamp","wlan.seq"]},
]
CFG = {"seed": 0, "n_shap": 2000, "alpha": 0.10, "nbins": 15,
       "drop_fractions": [0.10, 0.20, 0.30],
       "normal_fracs": {"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
       "family_fracs": {"train":0.60,"cal":0.20,"test_seen":0.20},
       "xgb": {"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,
               "colsample_bytree":0.9,"tree_method":"hist"},
       "fig_dir":"figures", "report_dir":"reports"}
for d in [CFG["fig_dir"], CFG["report_dir"]]: os.makedirs(d, exist_ok=True)
print("configured")

configured


In [6]:
# Dataset loader (self-contained) and helpers
def load_dataset(spec):
    dd = spec["data_dir"]; os.makedirs(dd, exist_ok=True)
    if spec["kind"] == "zenodo":
        if not glob.glob(dd+"/**/*.csv", recursive=True):
            meta = requests.get(f"https://zenodo.org/api/records/{spec['record']}", timeout=60).json()
            for f in meta.get("files", []):
                n,u = f["key"], f["links"]["self"]
                if n.lower().endswith((".csv",".zip",".gz")):
                    open(os.path.join(dd,n),"wb").write(requests.get(u,timeout=1200).content)
            for z in glob.glob(dd+"/*.zip"): zipfile.ZipFile(z).extractall(dd)
        df = load_csvs(dd); lc,nv,fams = detect_schema(df, spec["label_col"], spec["normal_value"])
    else:
        pq = spec.get("parquet")
        if pq and os.path.exists(pq): df = pd.read_parquet(pq)
        else:
            df = pd.read_csv(spec["file"], low_memory=False, encoding="latin-1")
            if pq:
                try: df.to_parquet(pq)
                except Exception as e: print("parquet skip:", e)
        lc,nv = spec["label_col"], spec["normal_value"]; fams = [v for v in df[lc].unique() if v!=nv]
    if spec.get("subsample_n") and len(df) > spec["subsample_n"]:
        frac = spec["subsample_n"]/len(df)
        df = df.groupby(lc, group_keys=False).sample(frac=frac, random_state=42).reset_index(drop=True)
    if spec.get("include_families"):
        df = df[df[lc].isin([nv]+list(spec["include_families"]))].reset_index(drop=True)
        fams = list(spec["include_families"])
    return df, lc, nv, fams

def make_fitter():
    return lambda Xtr, ytr, sd: xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                                  random_state=sd, **CFG["xgb"]).fit(Xtr, ytr)

def coverage_detail(p, y, qhat):
    p2 = np.column_stack([1-np.asarray(p), np.asarray(p)]); sets = p2 >= (1-qhat); y = np.asarray(y)
    inset = sets[np.arange(len(y)), y]
    return float(inset.mean()), float(np.mean([inset[y==k].mean() for k in np.unique(y)])), float(sets.sum(1).mean())

def evaluate(clf, X_cal, y_cal, X_shift, y_shift):
    p_cal = clf.predict_proba(X_cal)[:,1]; p_shift = clf.predict_proba(X_shift)[:,1]
    qhat = conformal_qhat(p_cal, y_cal, alpha=CFG["alpha"])
    _, bcov, _ = coverage_detail(p_shift, y_shift, qhat)
    ba = balanced_accuracy_score(y_shift, (p_shift>=0.5).astype(int))
    ece = top_label_ece(p_shift, y_shift, CFG["nbins"])
    return ba, bcov, ece
print("helpers ready")

helpers ready


In [ ]:
# TSFS experiment: for each held-out family, select stable features using ONLY seen
# families, then compare the baseline (all features) with TSFS on the unseen family.
fit = make_fitter()
val_rows, res_rows = [], []
for spec in DATASETS:
    df, lc, nv, fams = load_dataset(spec)
    for F in fams:
        S = prepare_splits(df, lc, nv, F, spec["drops"], CFG["normal_fracs"], CFG["family_fracs"], CFG["seed"])
        seen = [g for g in fams if g != F]
        nfeat = S["X_train"].shape[1]
        # (1) fragility estimated from SEEN families only
        internal = internal_fragility(fit, S["X_train"], S["y_train"], S["fam_train"],
                                      nv, seen, nfeat, n_shap=CFG["n_shap"], seed=CFG["seed"])
        # (2) oracle fragility on the true unseen family (for validation only)
        base = fit(S["X_train"], S["y_train"], CFG["seed"])
        ex = shap.TreeExplainer(base)
        def samp(X, m):
            idx = np.where(m)[0]
            if len(idx) > CFG["n_shap"]: idx = np.random.default_rng(CFG["seed"]).choice(idx, CFG["n_shap"], replace=False)
            return X[idx]
        m_ref = mean_shap(ex, samp(S["X_seen"], np.isin(S["fam_seen"], seen)))
        m_held = mean_shap(ex, samp(S["X_shift"], S["fam_shift"]==F))
        oracle = reversal_vector(m_ref, m_held)
        rho = spearmanr(internal, oracle).correlation
        k = max(3, int(0.1*nfeat)); ti=set(np.argsort(-internal)[:k]); to=set(np.argsort(-oracle)[:k])
        jac = len(ti & to)/len(ti | to)
        val_rows.append({"dataset": spec["name"], "held_out": str(F), "n_features": nfeat,
                         "rho_internal_oracle": round(float(rho),3), "topk_jaccard": round(jac,3)})
        # (3) baseline vs TSFS on the unseen family, across drop fractions
        ba_b, cov_b, ece_b = evaluate(base, S["X_cal"], S["y_cal"], S["X_shift"], S["y_shift"])
        for dfrac in CFG["drop_fractions"]:
            keep = select_stable_features(internal, dfrac)
            tclf = fit(S["X_train"][:,keep], S["y_train"], CFG["seed"])
            ba_t, cov_t, ece_t = evaluate(tclf, S["X_cal"][:,keep], S["y_cal"], S["X_shift"][:,keep], S["y_shift"])
            res_rows.append({"dataset": spec["name"], "held_out": str(F), "drop_fraction": dfrac,
                             "n_kept": len(keep),
                             "cov_baseline": round(cov_b,4), "cov_tsfs": round(cov_t,4), "cov_delta": round(cov_t-cov_b,4),
                             "balacc_baseline": round(ba_b,4), "balacc_tsfs": round(ba_t,4),
                             "ece_baseline": round(ece_b,4), "ece_tsfs": round(ece_t,4)})
        print(spec["name"], str(F), "done  (rho_int_oracle=%.2f)"%rho)
val = pd.DataFrame(val_rows); res = pd.DataFrame(res_rows)
val.to_csv(os.path.join(CFG["report_dir"],"09_tsfs_selection_validity.csv"), index=False)
res.to_csv(os.path.join(CFG["report_dir"],"09_tsfs_results.csv"), index=False)
print("done")

UAVIDS-2025 Sybil Attack done  (rho_int_oracle=0.18)
UAVIDS-2025 Blackhole Attack done  (rho_int_oracle=0.24)
UAVIDS-2025 Wormhole Attack done  (rho_int_oracle=0.54)
UAVIDS-2025 Flooding Attack done  (rho_int_oracle=0.47)
UAV-NIDD DDoS done  (rho_int_oracle=0.63)
UAV-NIDD UDP Flooding done  (rho_int_oracle=0.75)
UAV-NIDD MITM done  (rho_int_oracle=0.74)
UAV-NIDD Jamming done  (rho_int_oracle=0.67)
UAV-NIDD BruteForce done  (rho_int_oracle=0.71)


In [ ]:
# VALIDATION: can fragile features be identified without the unseen family?
print("Internal (seen-only) vs oracle (unseen) fragility agreement:")
print(val.to_string(index=False))
print("\nmean Spearman:", round(val["rho_internal_oracle"].mean(),3),
      " mean top-k Jaccard:", round(val["topk_jaccard"].mean(),3))

In [ ]:
# RESULT: does TSFS improve trust on the unseen family? (balanced conformal coverage)
piv = res.pivot_table(index=["dataset","held_out"], columns="drop_fraction",
                      values="cov_delta").round(4)
piv.columns = ["cov_delta@%.2f"%c for c in piv.columns]
print("Change in shift balanced-coverage, TSFS minus baseline (positive = TSFS better):")
print(piv.to_string())
print("\nMean coverage delta by drop fraction:")
print(res.groupby("drop_fraction")[["cov_delta"]].mean().round(4).to_string())
print("\nMean balanced-accuracy: baseline %.4f  tsfs %.4f"
      % (res["balacc_baseline"].mean(), res["balacc_tsfs"].mean()))

In [ ]:
# FIGURE: (a) internal vs oracle fragility agreement per family; (b) baseline vs TSFS coverage
best = res[res["drop_fraction"]==0.20]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].bar(range(len(val)), val["rho_internal_oracle"], color="#00695c")
ax[0].set_xticks(range(len(val))); ax[0].set_xticklabels([r["held_out"][:9] for _,r in val.iterrows()], rotation=35, ha="right", fontsize=7)
ax[0].axhline(0, color="gray", lw=.6); ax[0].set_ylabel("Spearman (internal vs oracle)")
ax[0].set_title("Fragility is identifiable from seen families alone")
x = np.arange(len(best)); w = 0.4
ax[1].bar(x-w/2, best["cov_baseline"], w, label="baseline (all features)", color="#c62828")
ax[1].bar(x+w/2, best["cov_tsfs"], w, label="TSFS (stable features)", color="#2e7d32")
ax[1].axhline(1-CFG["alpha"], ls="--", color="gray", lw=1)
ax[1].set_xticks(x); ax[1].set_xticklabels([r["held_out"][:9] for _,r in best.iterrows()], rotation=35, ha="right", fontsize=7)
ax[1].set_ylabel("shift balanced coverage"); ax[1].set_title("TSFS vs baseline under attack-family shift (drop=0.20)")
ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(CFG["fig_dir"],"09_tsfs.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit
!git add src/ reports/ figures/ notebooks/
!git commit -m "09 TSFS: Transfer-Stable Feature Selection module and experiment (seen-only fragility selection, validated vs oracle, baseline vs TSFS under attack-family shift)"
!git push origin main